# Training YOLO — Deteksi Parkir (3 kelas)
Kelas: `space-empty`, `space-occupied`, `illegal-parking`.
Dataset sudah di-merge & diseragamkan ke bbox oleh `scripts/merge_yolo_datasets.py`.

> Catatan: kelas `illegal-parking` sangat sedikit (~273 box). Cell terakhir memakai augmentasi agresif; pantau recall kelas illegal.

In [ ]:
# 1. Install (skip jika sudah ada)
%pip install -q ultralytics

In [ ]:
# 2. Setup: temukan data.yaml & pilih device otomatis
from pathlib import Path
import torch

# ROOT = folder project (parent dari notebooks/). Sesuaikan bila jalan di Colab.
ROOT = Path.cwd()
if not (ROOT / "dataset_merged").exists():
    ROOT = ROOT.parent  # jalan dari dalam notebooks/
DATA = ROOT / "dataset_merged" / "data.yaml"
assert DATA.exists(), f"data.yaml tidak ditemukan: {DATA}"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("data  :", DATA)
print("device:", DEVICE, "(" + (torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU") + ")")

In [ ]:
# 3. Train
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano; ganti yolov8s/m.pt untuk akurasi lebih tinggi

results = model.train(
    data=str(DATA),
    epochs=100,
    imgsz=640,
    batch=16,          # turunkan bila VRAM kurang; -1 = auto
    device=DEVICE,
    patience=20,       # early stop bila tak membaik 20 epoch
    project=str(ROOT / "runs"),
    name="parking_yolov8n",
    # augmentasi bantu kelas illegal-parking yang minim
    mosaic=1.0, mixup=0.1, copy_paste=0.1,
)

In [ ]:
# 4. Evaluasi di test split (per-kelas)
best = YOLO(results.save_dir + "/weights/best.pt")
metrics = best.val(data=str(DATA), split="test", device=DEVICE)
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)
for i, name in metrics.names.items():
    print(f"  {name:16} AP50={metrics.box.ap50[i]:.3f}")

In [ ]:
# 5. Prediksi contoh (visual sanity check)
sample = next((ROOT / "dataset_merged" / "test" / "images").iterdir())
best.predict(sample, save=True, conf=0.25, device=DEVICE)
print("hasil tersimpan di folder runs/.../predict")